# Micro-Hay gated residual TCN 08

This experiment tests a standard sparse gated residual/Mixture-of-Experts decomposition. The frozen GRU is the slow always-on expert. A causal TCN is the fast expert, multiplied by a learned sigmoid gate supervised with class-balanced focal loss. Gate and residual receive only frozen-GRU hidden features and packed spike input; no teacher or predicted physical state is fed back. The zero residual projection makes epoch 0 exactly GRU-MSE.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/kaggle/working/LearningSingleCompartiment')
if not (ROOT / 'pyproject.toml').exists():
    if ROOT.exists() and any(ROOT.iterdir()): raise RuntimeError(f'{ROOT} exists but is not the project')
    subprocess.check_call(['git', 'clone', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', 'main'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

## Existing inputs only
Attach `hay_micro_4c_event_enriched_v2.h5` and the original `gru_mse.pt` using **Add Input**. No dataset generation and no v6/v7 checkpoint are required.

In [ ]:
def find_kaggle_input(working_path, pattern, label):
    working_path = Path(working_path)
    candidates = [working_path] if working_path.exists() else []
    if Path('/kaggle/input').exists(): candidates += sorted(Path('/kaggle/input').rglob(pattern))
    if not candidates: raise FileNotFoundError(f'{label} non trovato. Montalo con Add Input: {pattern}')
    selected = candidates[0]
    print(f'{label}:', selected, f'({selected.stat().st_size/2**20:.1f} MiB)')
    return selected

DATASET = find_kaggle_input('/kaggle/working/hay_micro_4c_event_enriched_v2.h5', 'hay_micro_4c_event_enriched_v2*.h5', 'Dataset')
BASELINE = find_kaggle_input('/kaggle/working/gru_mse.pt', 'gru_mse.pt', 'Checkpoint GRU-MSE')
os.environ['HAY_FINETUNE_DATASET'] = str(DATASET)
os.environ['HAY_FINETUNE_BASELINE'] = str(BASELINE)
os.environ['HAY_FINETUNE_OUTPUT'] = '/kaggle/working/hay_micro_gated_residual_tcn_finetune_08'
os.environ['HAY_FINETUNE_OBJECTIVE'] = 'gated_residual_tcn_v8'
os.environ['HAY_FINETUNE_EPOCHS'] = '40'
os.environ['HAY_FINETUNE_PATIENCE'] = '28'
os.environ['HAY_FINETUNE_LEARNING_RATE'] = '3e-5'
os.environ['HAY_FINETUNE_CURRICULUM_EPOCHS'] = '18'
os.environ['HAY_FINETUNE_WINDOWS_PER_EPOCH'] = '48'
os.environ['HAY_FINETUNE_FORCE_RESTART'] = '0'
%run /kaggle/working/LearningSingleCompartiment/notebooks/micro_spike_finetune_03.py

## Gate, physical and 61-state audit
Gate metrics are diagnostics only; checkpoint selection still depends exclusively on physical rollout quality and the hard global/subthreshold/state constraints.

In [ ]:
import pandas as pd
from IPython.display import Image, display
RESULTS = Path('/kaggle/working/hay_micro_gated_residual_tcn_finetune_08')
display(pd.read_csv(RESULTS / 'comparison.csv').T)
history = pd.read_csv(RESULTS / 'finetune_history.csv')
columns = [c for c in ['epoch','event_scale','checkpoint_admissible','validation_selection','validation_global','validation_subthreshold_rmse_mV','validation_predicted_crossings','validation_waveform','validation_derivative','validation_gate_focal','validation_gate_target_fraction','validation_gate_predicted_fraction','validation_gate_true_positive_rate','validation_gate_false_positive_rate','validation_mean_state_normalized_rmse','validation_slow_state_mean_normalized_rmse','eta_s'] if c in history]
display(history[columns].tail(40))
statewise = pd.read_csv(RESULTS / 'statewise_rmse.csv')
ratio_columns = [c for c in statewise if c.endswith('_normalized_ratio_vs_gru_mse') and c != 'gru_mse_normalized_ratio_vs_gru_mse']
if ratio_columns: display(statewise.sort_values(ratio_columns[0], ascending=False)[['state','is_slow_state',*ratio_columns]].head(20))
display(Image(str(RESULTS / 'soma_comparison.png')))

## Export complete evidence
The ZIP contains selected and last checkpoints, history including gate diagnostics, test predictions, plot and all 61 state-wise errors.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, FileLink, display
zip_path = Path(make_archive('/kaggle/working/hay_micro_gated_residual_tcn_finetune_08_complete', 'zip', root_dir=RESULTS.parent, base_dir=RESULTS.name))
print('Archive:', zip_path, f'({zip_path.stat().st_size/2**20:.1f} MiB)')
display(FileLink(str(zip_path)))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{encoded}'),x=new Uint8Array(b.length);for(let i=0;i<b.length;i++)x[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([x],{{type:'application/zip'}})),a=document.createElement('a');a.href=u;a.download='{zip_path.name}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(u),60000);"""))
print('Download avviato:', zip_path.name)